# The-KISS Neural Cellular Automata (PyTorch Lightning)

This notebook reproduces the core experiment from `Growing_Neural_Cellular_Automata.ipynb`, but replaces the TensorFlow emoji target with 64×64 Gustav Klimt painting targets from `paintings/64`.  The reusable model, data, plotting, and training helpers live in `utils.py`, so this notebook contains only the experiment wiring.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from utils import (
    NCALightningModel,
    list_paintings,
    load_painting,
    make_step_loader,
    make_trainer,
    plot_loss,
    save_checkpoint,
    show_image,
    tensor_to_numpy_image,
    to_rgb,
    visualize_states,
)

torch.set_float32_matmul_precision("medium")


## Experiment parameters

The reference notebook offers growing, persistent, and regenerating modes.  The defaults below use the regenerating setup, which trains from a pattern pool and damages a few samples each batch.  Reduce `MAX_STEPS` while testing on CPU, then increase it for final results.


In [ ]:
# Cellular automata parameters inspired by the reference notebook.
CHANNEL_N = 16
TARGET_PADDING = 16
BATCH_SIZE = 8
POOL_SIZE = 1024
CELL_FIRE_RATE = 0.5

# Training parameters. 8000 matches the reference notebook; smaller values are useful for smoke tests.
MAX_STEPS = 8000
LEARNING_RATE = 2e-3

EXPERIMENT_TYPE = "Regenerating"  # choose: "Growing", "Persistent", or "Regenerating"
EXPERIMENT_MAP = {"Growing": 0, "Persistent": 1, "Regenerating": 2}
EXPERIMENT_N = EXPERIMENT_MAP[EXPERIMENT_TYPE]
USE_PATTERN_POOL = [False, True, True][EXPERIMENT_N]
DAMAGE_N = [0, 0, 3][EXPERIMENT_N]


## Choose a Klimt target painting

All files in `paintings/64` are available as targets.  Change `TARGET_PAINTING` to another file name from the printed list to train a different painting.


In [ ]:
painting_paths = list_paintings("paintings/64")
print("Available paintings:")
for path in painting_paths:
    print(" -", path.name)

TARGET_PAINTING = "the_kiss.png"
target_path = Path("paintings/64") / TARGET_PAINTING
target_rgba = load_painting(target_path, max_size=64)
show_image(target_rgba[..., :3] + (1.0 - target_rgba[..., 3:4]), scale=4, title=f"Target: {TARGET_PAINTING}")


## Build the Lightning model

`NCALightningModel` contains the PyTorch version of the NCA update rule and a Lightning manual-optimization training step.  Its training loop samples the pattern pool, resets the worst sample to the seed, optionally damages samples, runs a random 64–96 NCA steps, normalizes gradients, and commits the resulting states back to the pool.


In [ ]:
model = NCALightningModel(
    target_rgba=target_rgba,
    channel_n=CHANNEL_N,
    batch_size=BATCH_SIZE,
    pool_size=POOL_SIZE,
    target_padding=TARGET_PADDING,
    lr=LEARNING_RATE,
    fire_rate=CELL_FIRE_RATE,
    use_pattern_pool=USE_PATTERN_POOL,
    damage_n=DAMAGE_N,
)

print(model.ca)


## Train

Run the next cell to train.  Lightning will use a GPU automatically when one is available; otherwise it falls back to CPU.


In [ ]:
trainer = make_trainer(max_steps=MAX_STEPS, accelerator="auto")
trainer.fit(model, train_dataloaders=make_step_loader(MAX_STEPS))


## Inspect the learned growth

The model grows from a single live seed cell.  Increase `GROW_STEPS` to see later stages of the learned painting.


In [ ]:
GROW_STEPS = 256
model.eval()
with torch.no_grad():
    grown = model.grow(steps=GROW_STEPS, batch_size=1)

rgb = tensor_to_numpy_image(to_rgb(grown))[0]
show_image(rgb, scale=4, title=f"Grown {TARGET_PAINTING} after {GROW_STEPS} NCA steps")
plot_loss(model.loss_history)


## Visualize pool samples

For persistent/regenerating experiments, the pool contains partially grown states at many stages.


In [ ]:
if USE_PATTERN_POOL:
    visualize_states(model.pool[:49], title="Pattern pool snapshot", scale=2)


## Save weights

The checkpoint stores the PyTorch state dict and the hyperparameters needed to reconstruct the model.


In [ ]:
checkpoint_path = Path("train_log") / f"{target_path.stem}_nca.pt"
save_checkpoint(model, checkpoint_path)
print(f"Saved {checkpoint_path}")
